In [11]:
# Cell 1 - Imports and Logging
import sys, logging, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
 
logging.basicConfig(level=logging.INFO,
    format="%(asctime)s | %(name)s | %(message)s")
 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
 
from src.analytics.db_connector import get_connection, populate_financials, query_financials
from src.analytics.data_combiner import merge_mysql_mongodb, demonstrate_join_types
 
print("Imports successful")


Imports successful


In [16]:
cleaned_df = pd.read_csv("../processed/cleaned/movies_clean.csv")
print(f"Loaded {len(cleaned_df)} cleaned rows")
print(cleaned_df.dtypes)
 
# Parse release_year from release_date if not already present
cleaned_df["release_year"] = pd.to_datetime(
    cleaned_df["release_date"], errors="coerce"
).dt.year.fillna(0).astype(int)
 
# Ensure numeric types
cleaned_df["budget_usd"] = pd.to_numeric(
    cleaned_df["budget_usd"], errors="coerce"
).fillna(0).astype(int)
cleaned_df["revenue_usd"] = pd.to_numeric(
    cleaned_df["revenue_usd"], errors="coerce"
).fillna(0).astype(int)

Loaded 126 cleaned rows
id                     int64
title                    str
release_date             str
release_year           int64
genre                    str
primary_genre            str
budget_usd             int64
revenue_usd            int64
vote_average         float64
vote_count             int64
popularity           float64
original_language        str
overview                 str
director                 str
country                  str
dtype: object


In [17]:
conn = get_connection(host="localhost", user="root", password="", database="movies")
 
cursor = conn.cursor()
cursor.execute("DELETE FROM movie_financials")
conn.commit()
cursor.execute("SELECT COUNT(*) FROM movie_financials")
print("Rows after delete:", cursor.fetchone()[0])
cursor.close()
 
# Insert from cleaned CSV
inserted, skipped = populate_financials(conn, cleaned_df)
print(f"Inserted: {inserted} | Skipped: {skipped}")

2026-05-04 19:02:14,329 | src.analytics.db_connector | Connected to MySQL database: movies
2026-05-04 19:02:14,356 | src.analytics.db_connector | Inserted 126 rows, skipped 0 rows


Rows after delete: 0
Inserted: 126 | Skipped: 0


In [18]:
mysql_df = query_financials(conn, sql="""
    SELECT id, title, budget_usd, revenue_usd, release_year, genre
    FROM movie_financials
    WHERE budget_usd > 0 AND revenue_usd > 0
""")
print(f"MySQL rows with budget and revenue: {len(mysql_df)}")
print(mysql_df[["id", "title", "release_year", "genre"]].head(10))
print(mysql_df["release_year"].isna().sum())


2026-05-04 19:02:17,565 | src.analytics.db_connector | Queried 126 rows from MySQL


MySQL rows with budget and revenue: 126
   id            title  release_year     genre
0   1        Inception          2010    Sci-Fi
1   2  The Dark Knight          2008    Action
2   3     Interstellar          2014    Sci-Fi
3   4         Parasite          2019  Thriller
4   5          Titanic          1997   Romance
5   6           Avatar          2009    Sci-Fi
6   7    The Godfather          1972     Crime
7   8     Pulp Fiction          1994     Crime
8   9       The Matrix          1999    Sci-Fi
9  10        Gladiator          2000    Action
0


In [19]:
# Cell 5 - Select metadata columns (simulating MongoDB source)
mongo_cols = ["id", "title", "vote_average", "vote_count",
              "popularity", "original_language", "overview"]
mongo_cols_available = [c for c in mongo_cols if c in cleaned_df.columns]
mongo_df = cleaned_df[mongo_cols_available].copy()
 
# Merge MySQL financials with metadata on id
combined_df = merge_mysql_mongodb(mysql_df, mongo_df, on="id", how="inner")
print(f"Combined DataFrame shape: {combined_df.shape}")
print(combined_df.columns.tolist())
combined_df.head(3)


2026-05-04 19:02:21,482 | src.analytics.data_combiner | Merging MySQL (126 rows) and MongoDB (126 rows) on "id" with how="inner"
2026-05-04 19:02:21,485 | src.analytics.data_combiner | Merged result: 126 rows, 12 columns


Combined DataFrame shape: (126, 12)
['id', 'title_mysql', 'budget_usd', 'revenue_usd', 'release_year', 'genre', 'title_mongo', 'vote_average', 'vote_count', 'popularity', 'original_language', 'overview']


,id,title_mysql,budget_usd,revenue_usd,release_year,genre,title_mongo,vote_average,vote_count,popularity,original_language,overview
0,1,Inception,160000000,829895144,2010,Sci-Fi,Inception,8.8,2400000,83.95,en,No overview available.
1,2,The Dark Knight,185000000,1004558444,2008,Action,The Dark Knight,9.0,2700000,123.17,en,No overview available.
2,3,Interstellar,165000000,677471339,2014,Sci-Fi,Interstellar,8.6,2100000,140.24,en,No overview available.


In [20]:
print("Join type comparison (rows returned):")
join_counts = demonstrate_join_types(mysql_df, mongo_df, on="id")
 
# Visualise the difference
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(join_counts.keys(), join_counts.values(),
       color=["#2E75B6", "#70AD47", "#FFC000", "#FF0000"])
ax.set_title("Row count by join type")
ax.set_ylabel("Number of rows")
plt.tight_layout()
plt.savefig("../data/processed/analytics/join_comparison.png", dpi=120)
plt.show()


Join type comparison (rows returned):
  inner  join ->   126 rows
  left   join ->   126 rows
  right  join ->   126 rows
  outer  join ->   126 rows


In [21]:
from src.analytics.pivot_builder import wide_to_long, build_pivot_table, build_crosstab
 
df = combined_df.copy()
 
# Resolve duplicate title columns from the merge
if "title_mysql" in df.columns:
    df["title"] = df["title_mysql"].fillna(df.get("title_mongo", ""))
 
# Extract first genre as primary_genre
if "genre" in df.columns:
    df["primary_genre"] = df["genre"].str.split(",").str[0].str.strip()
else:
    df["primary_genre"] = "Unknown"
 
print("Top genres in dataset:")
print(df["primary_genre"].value_counts().head(10))


Top genres in dataset:
primary_genre
Action       78
Horror       15
Sci-Fi       10
Comedy        6
Thriller      4
Romance       3
Crime         3
Adventure     2
Family        2
Drama         1
Name: count, dtype: int64


In [22]:
temp_df = df[["id", "title", "budget_usd", "revenue_usd", "popularity", "genre"]].copy()
temp_df["genre"] = temp_df["genre"].fillna("Unknown")
temp_df["popularity"] = temp_df["popularity"].fillna(0)
temp_df["budget_usd"] = temp_df["budget_usd"].fillna(0)
temp_df["revenue_usd"] = temp_df["revenue_usd"].fillna(0)
 
financial_long = wide_to_long(
    df=temp_df.dropna(subset=["id"]),
    id_vars=["id", "title"],
    value_vars=["budget_usd", "revenue_usd", "popularity"],
    var_name="metric",
    value_name="value"
)
print("Long format shape:", financial_long.shape)
print(financial_long.head(6))


2026-05-04 19:02:33,248 | src.analytics.pivot_builder | Wide->Long: 126 rows x 6 cols -> 378 rows x 4 cols


Long format shape: (378, 4)
   id            title      metric        value
0   1        Inception  budget_usd  160000000.0
1   2  The Dark Knight  budget_usd  185000000.0
2   3     Interstellar  budget_usd  165000000.0
3   4         Parasite  budget_usd   11400000.0
4   5          Titanic  budget_usd  200000000.0
5   6           Avatar  budget_usd  237000000.0


In [23]:
df["release_year"] = pd.to_numeric(df["release_year"], errors="coerce")
df = df.dropna(subset=["release_year"])
df = df[df["release_year"] > 0]
df["release_year"] = df["release_year"].astype(int)
df["decade"] = ((df["release_year"] // 10) * 10).astype(str) + "s"
 
print(df.shape)
print(df[["release_year", "decade"]].head())
 
pt_revenue = build_pivot_table(
    df=df,
    values="revenue_usd",
    index="release_year",
    columns="primary_genre",
    aggfunc="mean"
)
 
# Show only top 6 genres by average revenue_usd
top_genres = df.groupby("primary_genre")["revenue_usd"].mean().nlargest(6).index
pt_display = pt_revenue[top_genres]
print(pt_display.tail(10))
 
pt_display.to_csv("../data/processed/analytics/pivot_genre_year.csv")
print("Saved pivot_genre_year.csv")


2026-05-04 19:02:39,839 | src.analytics.pivot_builder | Pivot table shape: (31, 12)


(126, 15)
   release_year decade
0          2010  2010s
1          2008  2000s
2          2014  2010s
3          2019  2010s
4          1997  1990s
primary_genre     Adventure        Sci-Fi        Family      Romance  \
release_year                                                           
2016           0.000000e+00  0.000000e+00  0.000000e+00  105523733.0   
2017           0.000000e+00  1.483271e+08  0.000000e+00          0.0   
2018           2.048360e+09  0.000000e+00  0.000000e+00          0.0   
2019           0.000000e+00  0.000000e+00  0.000000e+00          0.0   
2020           0.000000e+00  0.000000e+00  0.000000e+00          0.0   
2021           0.000000e+00  1.901217e+09  0.000000e+00          0.0   
2023           0.000000e+00  0.000000e+00  1.362836e+09          0.0   
2024           0.000000e+00  0.000000e+00  0.000000e+00          0.0   
2025           0.000000e+00  0.000000e+00  0.000000e+00          0.0   
2026           8.348713e+08  4.286571e+08  5.556673e+08  169

In [24]:
# Cell 10 - Cross-tabulation
df["original_language"] = df["original_language"].fillna("Unknown")
df["decade"] = ((df["release_year"] // 10) * 10).astype(int).astype(str) + "s"
 
top_langs = df["original_language"].value_counts().head(5).index
df_lang = df[df["original_language"].isin(top_langs)]
 
ct = build_crosstab(df_lang, row_col="original_language", col_col="decade")
print("Cross-tabulation of language vs decade:")
print(ct)


Cross-tabulation of language vs decade:
decade             1950s  1960s  1970s  1980s  1990s  2000s  2010s  2020s  All
original_language                                                             
en                     1      1      3      2      5      6     12     89  119
es                     0      0      0      0      0      0      0      2    2
hi                     0      0      0      0      0      0      0      1    1
ko                     0      0      0      0      0      0      2      1    3
zh                     0      0      0      0      0      0      0      1    1
All                    1      1      3      2      5      6     14     94  126


In [25]:
# Genre summary
from src.analytics.aggregator import genre_summary, yearly_trends, top_n_per_group
 
genre_df = genre_summary(df.dropna(subset=["vote_average"]))
print("Genre summary (top 10 by movie count):")
print(genre_df.head(10).to_string(index=False))

2026-05-04 19:07:08,885 | src.analytics.aggregator | Genre summary: 12 genres


Genre summary (top 10 by movie count):
primary_genre  avg_rating  median_rating  movie_count  total_revenue  total_budget
       Action    7.062821         7.0600           78    32037470230   11596713581
       Horror    6.029600         5.8000           15      891938037     330158769
       Sci-Fi    7.880000         7.8450           10     9752510530    1688891023
       Comedy    6.572500         6.2910            6      319917731     221177347
     Thriller    6.707250         6.5410            4      638215747     129707455
        Crime    8.200000         8.9000            3      489586841      55658655
      Romance    6.890000         6.7700            3     2476717957     260115687
    Adventure    7.635500         7.6355            2     2883231057     511891325
       Family    7.182000         7.1820            2     1918503401     236157350
    Animation    6.940000         6.9400            1      345992331     171200427


In [26]:
genre_df.to_csv("../data/processed/analytics/genre_analysis.csv", index=False)

In [27]:
yearly_df = yearly_trends(df)
yearly_df = yearly_df[
    (yearly_df["release_year"] >= 1990) & (yearly_df["release_year"] <= 2024)
]
 
fig, axes = plt.subplots(2, 1, figsize=(12, 8))
 
axes[0].plot(yearly_df["release_year"], yearly_df["movie_count"],
             color="#2E75B6", linewidth=2)
axes[0].set_title("Number of Movies per Year")
axes[0].set_ylabel("Count")
 
axes[1].plot(yearly_df["release_year"], yearly_df["total_revenue"] / 1e6,
             color="#70AD47", linewidth=2)
axes[1].set_title("Total revenue_usd per Year (Millions USD)")
axes[1].set_ylabel("Revenue (M$)")
axes[1].set_xlabel("Year")
 
plt.tight_layout()
plt.savefig("../data/processed/analytics/yearly_trends.png", dpi=120)
plt.show()


2026-05-04 19:07:33,521 | src.analytics.aggregator | Yearly trends: 31 years


In [28]:
yearly_df.to_csv("../data/processed/analytics/yearly_trends.csv", index=False)

In [35]:
top_movies = top_n_per_group(df, group_col="primary_genre", sort_col="revenue_usd", n=3).reset_index(drop=True)

print("Top 3 movies per genre by revenue_usd:")
print(top_movies[["title", "revenue_usd"]].head(3).to_string(index=False))

2026-05-04 19:11:43,538 | src.analytics.aggregator | top_n_per_group: group=primary_genre sort=revenue_usd n=3 -> 28 rows


Top 3 movies per genre by revenue_usd:
          title  revenue_usd
  Countess Dora   1014438485
The Dark Knight   1004558444
     Rust Creek    932382278


In [36]:
from src.analytics.time_series import (parse_release_dates, build_monthly_series, resample_series, rolling_averages)
 
df_dated = parse_release_dates(cleaned_df, date_col="release_date")
 
print("Date columns added:")
print(df_dated[["title", "release_date", "release_year", "release_month", "release_weekday", "release_quarter"]].head(5))
print(f"Valid dates: {df_dated['release_date'].notna().sum()}")
print(f"NaT values:  {df_dated['release_date'].isna().sum()}")

2026-05-04 19:16:52,217 | src.analytics.time_series | Parsed 126 / 126 dates from "release_date"


Date columns added:
                            title release_date  release_year  release_month  \
0                      Zootopia 2   2025-11-26          2025             11   
1                 Alien: Covenant   2017-05-09          2017              5   
2                           David   2025-12-14          2025             12   
3  Good Luck, Have Fun, Don't Die   2026-02-13          2026              2   
4                        Solo Mio   2026-02-05          2026              2   

   release_weekday  release_quarter  
0                2                4  
1                1                2  
2                6                4  
3                4                1  
4                3                1  
Valid dates: 126
NaT values:  0


In [38]:
monthly_revenue = build_monthly_series(df_dated, date_col="release_date", value_col="revenue_usd")
print("Monthly revenue_usd series (last 12 months):")
print(monthly_revenue.tail(12))


2026-05-04 19:17:10,802 | src.analytics.time_series | Monthly series: 835 periods, total=52150474336


Monthly revenue_usd series (last 12 months):
release_date
2026-01-31    4564778206
2026-02-28    4242309795
2026-03-31    4956196387
2026-04-30    1352332890
2026-05-31             0
2026-06-30             0
2026-07-31     506692126
2026-08-31             0
2026-09-30             0
2026-10-31     345992331
2026-11-30             0
2026-12-31     173876677
Freq: ME, Name: revenue_usd, dtype: int64


In [39]:
# Resample to yearly
yearly_revenue = resample_series(monthly_revenue, freq="YE", agg="sum")
print("\nYearly revenue_usd totals:")
print(yearly_revenue.tail(10))

2026-05-04 19:17:14,757 | src.analytics.time_series | Resampled to freq=YE agg=sum -> 70 periods



Yearly revenue_usd totals:
release_date
2017-12-31      148327117
2018-12-31     2048359754
2019-12-31     1191182278
2020-12-31      650160121
2021-12-31     2212985143
2022-12-31              0
2023-12-31     1377836132
2024-12-31      430912505
2025-12-31     9638368157
2026-12-31    16142178412
Freq: YE-DEC, Name: revenue_usd, dtype: int64


In [40]:
# Rolling averages on monthly revenue_usd
rolling_df = rolling_averages(monthly_revenue, windows=(3, 6, 12))
 
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(rolling_df.index, rolling_df["value"] / 1e9,
        alpha=0.3, color="gray", label="Monthly total")
ax.plot(rolling_df.index, rolling_df["rolling_3"] / 1e9,
        color="#2E75B6", linewidth=1.5, label="3-month rolling avg")
ax.plot(rolling_df.index, rolling_df["rolling_12"] / 1e9,
        color="#FF0000", linewidth=2, label="12-month rolling avg")
ax.set_title("Monthly revenue_usd with Rolling Averages (Billions USD)")
ax.set_ylabel("Revenue ($B)")
ax.legend()
plt.tight_layout()
plt.savefig("../data/processed/analytics/rolling_revenue.png", dpi=120)
plt.show()


2026-05-04 19:17:22,805 | src.analytics.time_series | Rolling average window=3 computed
2026-05-04 19:17:22,810 | src.analytics.time_series | Rolling average window=6 computed
2026-05-04 19:17:22,811 | src.analytics.time_series | Rolling average window=12 computed


In [41]:
from pymongo import MongoClient
from src.analytics.mongo_pipeline import build_genre_pipeline, run_pipeline
 
client = MongoClient("mongodb://localhost:27017/")
collection = client["movie_db"]["movies"]
 
# Build pipeline: $match -> $addFields -> $unwind -> $group -> $sort -> $project
pipeline = build_genre_pipeline(min_vote_count=50, top_n=10)
genre_pipeline_df = run_pipeline(collection, pipeline)
 
print("Top 10 genres by avg rating (MongoDB pipeline result):")
print(genre_pipeline_df.to_string(index=False))

2026-05-04 19:19:13,025 | src.analytics.mongo_pipeline | Built genre pipeline: min_vote_count=50, top_n=10
2026-05-04 19:19:13,075 | src.analytics.mongo_pipeline | Pipeline returned 0 documents


Top 10 genres by avg rating (MongoDB pipeline result):
Empty DataFrame
Columns: []
Index: []


In [42]:
#Answer all four analytical questions
from src.analytics.insight_reporter import run_all_questions
 
# Add primary_genre to the date-parsed DataFrame
df_dated["primary_genre"] = (
    df_dated["genre"].str.split(",").str[0].str.strip()
    if "genre" in df_dated.columns
    else "Unknown"
)
 
# Merge in financial data if not already present
if "budget_usd" not in df_dated.columns or df_dated["budget_usd"].isna().all():
    df_analysis = df_dated.merge(
        mysql_df[["id", "budget_usd", "revenue_usd"]],
        on="id", how="left"
    )
else:
    df_analysis = df_dated.copy()
 
results = run_all_questions(df_analysis)


2026-05-04 19:22:31,072 | src.analytics.insight_reporter | Q1 top genres by revenue: 10 rows
2026-05-04 19:22:31,075 | src.analytics.insight_reporter | Q2 ROI by genre: 12 genres
2026-05-04 19:22:31,077 | src.analytics.insight_reporter | Q3 yearly volume: 31 years
2026-05-04 19:22:31,078 | src.analytics.insight_reporter | Q4 language distribution: 5 languages



ANALYTICAL FINDINGS SUMMARY

Q1 - Top Genre by Revenue: Adventure (avg $1,441,615,528)
Q2 - Best ROI Genre: Crime (22.8x average return)
Q3 - Peak Release Year: 2026 (56 movies)
Q4 - Dominant Language: en (119 movies, 94.4% of dataset)



In [45]:
# Visualise ROI by genre
roi_df = results["roi_by_genre"].reset_index()
top10_roi = roi_df.head(10)
 
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top10_roi["primary_genre"], top10_roi["roi"], color="#2E75B6")
ax.set_title("Top 10 Movie Genres by ROI (revenue_usd / budget_usd)")
ax.set_xlabel("Return on Investment (x)")
ax.invert_yaxis()
 
for bar, val in zip(bars, top10_roi["roi"]):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f"{val:.1f}x", va="center", fontsize=9)
 
plt.tight_layout()
plt.savefig("../data/processed/analytics/genre_roi.png", dpi=120)
plt.show()
